# TuringBench + Defactify: Qwen-2.5-0.5B Fine-tuning
Fine-tune `Qwen/Qwen2.5-0.5B` on TuringBench + Defactify (frontier LLMs) for human-vs-AI detection.
Uses FP16 mixed-precision training on Kaggle T4 GPU. Pushes checkpoints to HuggingFace Hub.

In [ ]:
import os
import shutil

# Read HuggingFace token from Kaggle Secrets so it is never hardcoded.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print('Could not load HF_TOKEN from Kaggle Secrets:', e)
    print('Set HF_TOKEN manually or add it via Add-ons -> Secrets.')

repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/kaggle/working/ai-detection-at-scale'
# Force a fresh clone so we always use the latest fixed script.
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone $repo_url $repo_dir
print('Repo ready at', repo_dir)
!git -C $repo_dir log -1 --oneline

In [ ]:
# Install any missing deps
!pip install -q datasets transformers accelerate scikit-learn huggingface_hub packaging

In [ ]:
import os

script = os.path.join(repo_dir, 'scripts', '33_finetune_turingbench.py')
out_dir = '/kaggle/working/models/turingbench_qwen_0.5b'
os.makedirs(out_dir, exist_ok=True)

hub_model_id = 'vedangvatsa/vedang-turingbench-qwen-2.5-0.5b'

# Resume from the latest Hub checkpoint if one exists and HF_TOKEN is set.
resume_from = None
if os.environ.get('HF_TOKEN'):
    try:
        from huggingface_hub import list_repo_refs
        refs = list(list_repo_refs(repo_id=hub_model_id).branches)
        if refs:
            resume_from = hub_model_id
            print('Resuming from Hub checkpoint:', resume_from)
    except Exception as e:
        print('No existing Hub checkpoint found, starting fresh:', e)
else:
    print('HF_TOKEN not set; checkpoint resume disabled.')

# Qwen-2.5-0.5B works well with FP16 on T4 GPUs.
cmd = (
    f"python {script} "
    f"--model_name Qwen/Qwen2.5-0.5B "
    f"--output_dir {out_dir} "
    f"--max_length 512 "
    f"--epochs 3 "
    f"--batch_size 16 "
    f"--gradient_accumulation_steps 2 "
    f"--learning_rate 2e-5 "
    f"--fp16 "
    f"--hub_model_id {hub_model_id} "
)
if resume_from:
    cmd += f"--resume_from_checkpoint {resume_from} "
cmd += "--seed 42"

print('Running:', cmd)
!{cmd}

In [ ]:
import shutil
results_src = os.path.join(repo_dir, 'results', 'turingbench_finetuned.csv')
results_dst = '/kaggle/working/results/turingbench_finetuned.csv'
os.makedirs('/kaggle/working/results', exist_ok=True)
if os.path.exists(results_src):
    shutil.copy(results_src, results_dst)
    print('Results copied to', results_dst)
print('Outputs in /kaggle/working/models/turingbench_qwen_0.5b')